# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatima-zehra5/ML-internhip/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row represents one content item for one client on one report date in the daily performance table.

**Time window:** For this contract, I will work on the mid-panel month **March 2026** (`2026-03`) for verification and feature development. The warehouse overall spans 2025-01-27 to 2026-06-30, but June 2026 is treated as a sealed final month and is not used to develop the label logic.

**Decision moment:** The features must use information available before the outcome period, so the future impression outcome is kept separate from the feature window.

In [8]:
# Warehouse query setup
%pip -q install duckdb

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("DuckDB ready.")
print("Using March 2026 as the development month.")

DuckDB ready.
Using March 2026 as the development month.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features:** Previous-period impressions, query visibility, query concentration (top-query share), and ranking-position volatility. These are signals available before the outcome period.

**Label:** Impression decline — latest-period impressions are less than 80% of previous-period impressions.

**Context:** `client_hash_id`, report date, and month are used only to define the client/time context and validation or slicing.

**Excluded:** Client names, domains/URLs, private queries, credentials, and any fields that directly reveal client identity are excluded because the analysis must remain anonymized and safe for decision-support.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:
# ML-04 — Load March 2026 warehouse partition

!pip -q install -U huggingface_hub duckdb

from google.colab import userdata
from huggingface_hub import hf_hub_download
import duckdb

# Read the token securely from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. Add/update HF_TOKEN in Colab Secrets "
        "and enable notebook access."
    )

# Download the March 2026 partition
march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print("March 2026 partition downloaded successfully.")

# Open DuckDB
con = duckdb.connect()

# Create the local working table
con.execute(f"""
CREATE OR REPLACE TABLE fact_content_daily_performance AS
SELECT *
FROM read_parquet('{march_path}')
""")

print("March 2026 warehouse table loaded successfully.")

# Show the actual columns
con.sql("""
DESCRIBE fact_content_daily_performance
""").df()

March 2026 partition downloaded successfully.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March 2026 warehouse table loaded successfully.


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


### Verification approach

I verify the contract on the March 2026 development month. The three checks cover:

1. **Grain:** confirm that the selected identifiers form the intended content-by-client-by-date grain.
2. **Slice size and date span:** measure the March 2026 row count and observed report-date range.
3. **Availability:** count rows where the required availability field is `IS TRUE`, so unavailable rows are not silently treated as usable data.

The results are used as measured checks on this slice and are not treated as causal evidence.


In [10]:
import duckdb

# Verification 1 — grain
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT(
        CAST(client_hash_id AS VARCHAR), '|',
        CAST(content_hash_id AS VARCHAR), '|',
        CAST(report_date AS VARCHAR)
    )) AS unique_client_content_date_rows
FROM fact_content_daily_performance
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_client_content_date_rows
0,9841378,9841378


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

This slice has several important limits. History is not perfectly balanced across all content items and clients, so observed patterns may reflect differences in coverage and available history. Search Console data can also have earlier or uneven availability across items, so missingness should not automatically be interpreted as zero performance. Rolling or lookback windows can overlap across report periods, which means adjacent observations may not be fully independent.

This data can support directional, measured decision-support signals, but it cannot establish causation, explain every reason for an impressions decline, or predict a search-engine algorithm refresh. Client-identifying information, domains, URLs, private queries, and credentials are deliberately excluded.


In [11]:
# Basic limitation check: March 2026 row count and date coverage
con.sql("""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM fact_content_daily_performance
""").df()

,row_count,min_report_date,max_report_date
0,9841378,2026-03-01,2026-03-31


## Self-check

Before you submit, confirm each line honestly:

- [yes  ] Every section above is filled — markdown thinking AND the code that backs it
- [yes ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ yes] No client names, URLs, or private queries anywhere
- [ yes] My claims use careful words: observed, measured, directional, decision-support
- [ yes] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.